# SimpleParT Benchmark Metrics

Notebook reads benchmark JSON logs and renders metric plots.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'artifacts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

LOG_DIR = PROJECT_ROOT / 'artifacts' / 'logs'
OUTPUT_PNG = LOG_DIR / 'benchmark_metrics.png'

LOG_DIR

In [ ]:
def load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def metric_value(data, *path):
    current = data
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return None
        current = current[key]
    if current in (None, ''):
        return None
    return float(current)


def runtime_record(runtime, data):
    if not data or data.get('available') is False or 'metrics' not in data:
        return None
    return {
        'runtime': runtime,
        'accuracy': metric_value(data, 'metrics', 'accuracy'),
        'loss': metric_value(data, 'metrics', 'loss'),
        'latency_mean_ms': metric_value(data, 'latency', 'mean_ms'),
        'latency_p95_ms': metric_value(data, 'latency', 'p95_ms'),
        'throughput_events_per_s': metric_value(data, 'latency', 'throughput_events_per_s'),
        'peak_rss_mb': metric_value(data, 'memory', 'peak_rss_mb'),
    }


logs = {
    'PyTorch': load_json(LOG_DIR / 'benchmark_pytorch.json'),
    'ONNX Runtime': load_json(LOG_DIR / 'benchmark_onnx.json'),
    'SOFIE': load_json(LOG_DIR / 'benchmark_sofie.json'),
}
records = [record for runtime, data in logs.items() if (record := runtime_record(runtime, data))]
records

In [ ]:
metric_specs = [
    ('Accuracy', 'accuracy', ''),
    ('Loss', 'loss', ''),
    ('Mean latency', 'latency_mean_ms', 'ms'),
    ('P95 latency', 'latency_p95_ms', 'ms'),
    ('Throughput', 'throughput_events_per_s', 'events/s'),
    ('Peak RSS', 'peak_rss_mb', 'MB'),
]
colors = {
    'PyTorch': '#335c67',
    'ONNX Runtime': '#e09f3e',
    'SOFIE': '#7a9e7e',
}

plt.rcParams['font.family'] = 'DejaVu Sans'
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
fig.suptitle('SimpleParT Runtime Benchmark Metrics', fontsize=18, fontweight='bold')

runtimes = [record['runtime'] for record in records]
for ax, (title, key, unit) in zip(axes.flatten(), metric_specs):
    values = [record.get(key) for record in records]
    numeric_values = [0.0 if value is None else float(value) for value in values]
    bars = ax.bar(runtimes, numeric_values, color=[colors.get(runtime, '#6c757d') for runtime in runtimes])
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='y', alpha=0.22)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', rotation=18)
    if unit:
        ax.set_ylabel(unit)

    for bar, value in zip(bars, values):
        if value is None:
            label = 'n/a'
        elif abs(float(value)) >= 1000:
            label = f'{float(value):,.0f}'
        elif abs(float(value)) >= 10:
            label = f'{float(value):.2f}'
        else:
            label = f'{float(value):.4f}'
        ax.annotate(
            label,
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=9,
        )

fig.tight_layout()
OUTPUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PNG, dpi=220, bbox_inches='tight')
OUTPUT_PNG